# SMS Spam Detection — Machine Learning Experiment

Notebook chính của dự án. Thực hiện lần lượt: **đọc dữ liệu → làm sạch → chia Train/Test → TF-IDF → huấn luyện → đánh giá → xuất kết quả → thử dự đoán**.

- Bộ dữ liệu huấn luyện: **UCI SMS Spam Collection (English)**.
- `app.py` chỉ tải Pipeline đã lưu; không cần chạy Notebook mỗi lần mở giao diện.
- Không sử dụng tập Test để huấn luyện hoặc chọn siêu tham số.

In [2]:
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Notebook đặt tại thư mục gốc dự án.
BASE_DIR = Path.cwd().resolve()
DATA_PATH = BASE_DIR / "data" / "sms+spam+collection" / "SMSSpamCollection"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"
assert DATA_PATH.is_file(), f"Không tìm thấy dataset: {DATA_PATH}. Hãy mở Notebook từ thư mục gốc dự án."
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", BASE_DIR)

Project: C:\ML\BTL\sms-spam-detection-converted


## 1. Đọc và làm sạch dữ liệu

Loại bỏ các hàng trùng hoàn toàn, không sửa nội dung SMS và nhãn gốc.

In [3]:
def load_dataset():
    """
    Đọc dataset SMS Spam Collection.

    Returns:
        DataFrame gồm 2 cột: label va message.
    """
    df = pd.read_csv(
        DATA_PATH,
        sep="\t",
        header=None,
        names=["label", "message"],
        encoding="utf-8"
    )

    return df


def clean_dataset(df):
    """
    Loai bo cac dong trung lap hoan toan.
    Khong thay doi DataFrame goc.

    Args:
        df: DataFrame chua du lieu SMS.

    Returns:
        DataFrame da loai bo trung lap.
    """
    df_clean = df.drop_duplicates().copy()

    # Danh lai chi so dong tu 0
    df_clean = df_clean.reset_index(drop=True)

    return df_clean


def split_dataset(df, test_size=0.2, random_state=42):
    """
    Chia du lieu thanh tap train va test.

    Args:
        df: DataFrame da lam sach.
        test_size: Ty le du lieu danh cho tap test.
        random_state: Co dinh ket qua chia du lieu.

    Returns:
        X_train, X_test, y_train, y_test
    """

    # X: Noi dung tin nhan
    X = df["message"]

    # y: Nhan phan loai (ham hoac spam)
    y = df["label"]

    # Chia du lieu theo ty le 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

In [4]:
df = load_dataset()
df_clean = clean_dataset(df)
print(f"Ban đầu: {len(df)} | Sau loại trùng: {len(df_clean)} | Đã loại: {len(df)-len(df_clean)}")
print("Phân bố nhãn:", df_clean['label'].value_counts())
display(df_clean.head())

Ban đầu: 5572 | Sau loại trùng: 5169 | Đã loại: 403
Phân bố nhãn: label
ham     4516
spam     653
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 2. Chia dữ liệu Train/Test

Chia 80/20, phân tầng theo nhãn (`stratify=y`), cố định `random_state=42`.

In [5]:
X_train, X_test, y_train, y_test = split_dataset(df_clean)
print("Train:", len(X_train), "| Test:", len(X_test))
print("Train labels:", y_train.value_counts())
print("Test labels:", y_test.value_counts())

Train: 4135 | Test: 1034
Train labels: label
ham     3613
spam     522
Name: count, dtype: int64
Test labels: label
ham     903
spam    131
Name: count, dtype: int64


## 3. TF-IDF và cấu hình hai Pipeline

TF-IDF sử dụng unigram/bigram, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`. Mỗi Pipeline tự học TF-IDF **chỉ trên Train** khi gọi `fit`, tránh rò rỉ dữ liệu Test.

In [6]:
def create_tfidf():
    """
    Khoi tao bo trich xuat dac trung TF-IDF.

    Returns:
        TfidfVectorizer chua duoc huan luyen.
    """
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )

    return vectorizer


def create_pipelines():
    """
    Khoi tao hai Machine Learning Pipeline:
    1. TF-IDF + Multinomial Naive Bayes
    2. TF-IDF + Logistic Regression

    Returns:
        Dictionary chua hai Pipeline chua huan luyen.
    """
    pipelines = {
        "Multinomial Naive Bayes": Pipeline([
            ("tfidf", create_tfidf()),
            ("classifier", MultinomialNB())
        ]),

        "Logistic Regression": Pipeline([
            ("tfidf", create_tfidf()),
            ("classifier", LogisticRegression(
                max_iter=1000,
                random_state=42
            ))
        ])
    }

    return pipelines


def train_models(pipelines, X_train, y_train):
    """
    Huan luyen cac Machine Learning Pipeline.

    Args:
        pipelines: Dictionary chua hai Pipeline.
        X_train: Noi dung SMS cua tap Train.
        y_train: Nhan Ham/Spam cua tap Train.

    Returns:
        Dictionary chua cac Pipeline da huan luyen.
    """
    trained_models = {}

    for name, pipeline in pipelines.items():
        print(f"\nDang huan luyen: {name}")

        # Pipeline tu dong chay TF-IDF roi huan luyen bo phan loai
        pipeline.fit(X_train, y_train)

        trained_models[name] = pipeline

        print(f"Huan luyen {name} thanh cong!")

    return trained_models


def save_models(trained_models):
    """
    Luu cac Pipeline da huan luyen vao thu muc models/.

    Args:
        trained_models: Dictionary chua hai Pipeline da huan luyen.
    """

    # Xac dinh duong dan thu muc models
    base_dir = BASE_DIR
    models_dir = base_dir / "models"

    # Tao thu muc neu chua ton tai
    models_dir.mkdir(parents=True, exist_ok=True)

    # Dat ten file cho tung mo hinh
    file_names = {
        "Multinomial Naive Bayes": "naive_bayes_pipeline.joblib",
        "Logistic Regression": "logistic_regression_pipeline.joblib"
    }

    # Luu tung Pipeline
    for name, model in trained_models.items():
        file_path = models_dir / file_names[name]

        joblib.dump(model, file_path)

        print(f"Da luu mo hinh {name}: {file_path}")

In [7]:
# Khảo sát kích thước đặc trưng trên Train; KHÔNG fit trên Test.
preview_vectorizer = create_tfidf()
X_train_tfidf = preview_vectorizer.fit_transform(X_train)
X_test_tfidf = preview_vectorizer.transform(X_test)
print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF:", X_test_tfidf.shape)
print("Vocabulary size:", len(preview_vectorizer.vocabulary_))
pipelines = create_pipelines()
for name, pipeline in pipelines.items():
    print(name, pipeline)

Train TF-IDF: (4135, 10188)
Test TF-IDF: (1034, 10188)
Vocabulary size: 10188
Multinomial Naive Bayes Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('classifier', MultinomialNB())])
Logistic Regression Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2),
                                 sublinear_tf=True)),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


## 4. Huấn luyện

Huấn luyện Multinomial Naive Bayes và Logistic Regression theo cấu hình gốc.

In [8]:
trained_models = train_models(pipelines, X_train, y_train)
print("Số mô hình:", len(trained_models))


Dang huan luyen: Multinomial Naive Bayes
Huan luyen Multinomial Naive Bayes thanh cong!

Dang huan luyen: Logistic Regression
Huan luyen Logistic Regression thanh cong!
Số mô hình: 2


## 5. Đánh giá và trực quan hóa

Các chỉ số Precision, Recall và F1 sử dụng **Spam là lớp dương**; ma trận nhầm lẫn theo thứ tự `[ham, spam]`.

In [9]:
def evaluate_model(model, X_test, y_test):
    """
    Danh gia mo hinh tren tap Test.

    Args:
        model: Pipeline da duoc huan luyen.
        X_test: Noi dung SMS cua tap Test.
        y_test: Nhan thuc te cua tap Test.

    Returns:
        Dictionary chua cac chi so danh gia.
    """

    # Du doan nhan Ham/Spam tren tap Test
    y_pred = model.predict(X_test)

    # Tinh cac chi so, xem Spam la lop duong tinh
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(
            y_test, y_pred, pos_label="spam", zero_division=0
        ),
        "recall": recall_score(
            y_test, y_pred, pos_label="spam", zero_division=0
        ),
        "f1_score": f1_score(
            y_test, y_pred, pos_label="spam", zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            y_test, y_pred, labels=["ham", "spam"]
        )
    }

    return metrics


def plot_confusion_matrix(cm, model_name, output_path):
    """
    Ve va luu Confusion Matrix cua mo hinh.

    Args:
        cm: Ma tran nham lan tu evaluate_model().
        model_name: Ten mo hinh.
        output_path: Duong dan file anh PNG.
    """

    # Tao thu muc dau ra neu chua ton tai
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Tao khung bieu do
    plt.figure(figsize=(6, 5))

    # Ve heatmap Confusion Matrix
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Ham", "Spam"],
        yticklabels=["Ham", "Spam"]
    )

    # Them tieu de va nhan truc
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    # Luu bieu do
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")

    # Dong bieu do de giai phong bo nho
    plt.close()

    print(f"Da luu bieu do: {output_path}")


def plot_model_comparison(metrics_df, output_path):
    """
    Ve va luu bieu do so sanh hieu suat cac mo hinh.

    Args:
        metrics_df: DataFrame chua cac chi so danh gia.
        output_path: Duong dan luu bieu do PNG.
    """

    # Chuyen duong dan sang doi tuong Path
    output_path = Path(output_path)

    # Tao thu muc neu chua ton tai
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Chuyen du lieu sang dang phu hop de ve bieu do
    plot_df = metrics_df.melt(
        id_vars="model",
        value_vars=[
            "accuracy",
            "precision",
            "recall",
            "f1_score"
        ],
        var_name="Metric",
        value_name="Score"
    )

    # Tao khung bieu do
    plt.figure(figsize=(10, 6))

    # Ve bieu do cot so sanh
    ax = sns.barplot(
        data=plot_df,
        x="Metric",
        y="Score",
        hue="model"
    )

    # Hien thi gia tri tren tung cot
    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.3f",
            padding=3,
            fontsize=9
        )

    # Thiet lap tieu de va nhan truc
    plt.title("Model Performance Comparison")
    plt.xlabel("Evaluation Metric")
    plt.ylabel("Score")
    plt.ylim(0, 1.1)
    plt.legend(title="Model")

    # Luu bieu do
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    # Dong bieu do
    plt.close()

    print(f"Da luu bieu do so sanh: {output_path}")

In [10]:
metrics_records = []
for name, model in trained_models.items():
    results = evaluate_model(model, X_test, y_test)
    metrics_records.append({
        "model": name,
        "accuracy": results["accuracy"],
        "precision": results["precision"],
        "recall": results["recall"],
        "f1_score": results["f1_score"],
    })
    print(f"\n{name}: Accuracy={results['accuracy']:.4f}, Precision={results['precision']:.4f}, Recall={results['recall']:.4f}, F1={results['f1_score']:.4f}")
    print("Confusion Matrix [ham, spam]:\n", results["confusion_matrix"])
    image_name = ("confusion_matrix_naive_bayes.png" if name == "Multinomial Naive Bayes"
                  else "confusion_matrix_logistic_regression.png")
    plot_confusion_matrix(results["confusion_matrix"], name, OUTPUTS_DIR / image_name)

metrics_df = pd.DataFrame(metrics_records)
metrics_df.to_csv(OUTPUTS_DIR / "metrics.csv", index=False)
plot_model_comparison(metrics_df, OUTPUTS_DIR / "model_comparison.png")
display(metrics_df)


Multinomial Naive Bayes: Accuracy=0.9632, Precision=1.0000, Recall=0.7099, F1=0.8304
Confusion Matrix [ham, spam]:
 [[903   0]
 [ 38  93]]
Da luu bieu do: C:\ML\BTL\sms-spam-detection-converted\outputs\confusion_matrix_naive_bayes.png

Logistic Regression: Accuracy=0.9623, Precision=1.0000, Recall=0.7023, F1=0.8251
Confusion Matrix [ham, spam]:
 [[903   0]
 [ 39  92]]
Da luu bieu do: C:\ML\BTL\sms-spam-detection-converted\outputs\confusion_matrix_logistic_regression.png
Da luu bieu do so sanh: C:\ML\BTL\sms-spam-detection-converted\outputs\model_comparison.png


,model,accuracy,precision,recall,f1_score
0,Multinomial Naive Bayes,0.963250,1.0,0.709924,0.830357
1,Logistic Regression,0.962282,1.0,0.702290,0.825112


**Nhận xét:** Cần xem đồng thời Precision và Recall của lớp Spam. Precision cao không đồng nghĩa phát hiện được tất cả Spam; Recall thấp tương ứng với nhiều tin Spam bị bỏ sót. Kết quả thực tế nằm trong bảng trên.

## 6. Lưu Pipeline và thử dự đoán

Lưu **Pipeline gồm TF-IDF + classifier** để `app.py` dự đoán trực tiếp từ văn bản gốc.

In [11]:
save_models(trained_models)

def load_model(model_name="naive_bayes"):
    model_files = {
        "naive_bayes": "naive_bayes_pipeline.joblib",
        "logistic_regression": "logistic_regression_pipeline.joblib",
    }
    if model_name not in model_files:
        raise ValueError(f"Mô hình không hợp lệ: {model_name}")
    return joblib.load(MODELS_DIR / model_files[model_name])

def predict_message(model, message):
    prediction = model.predict([message])[0]
    probabilities = model.predict_proba([message])[0]
    return prediction, dict(zip(model.classes_, probabilities))

test_message = "Congratulations! You have won a free prize. Claim now!"
for model_name in ["naive_bayes", "logistic_regression"]:
    prediction, probabilities = predict_message(load_model(model_name), test_message)
    print(f"{model_name}: {prediction} | Ham={probabilities['ham']:.4f} | Spam={probabilities['spam']:.4f}")

Da luu mo hinh Multinomial Naive Bayes: C:\ML\BTL\sms-spam-detection-converted\models\naive_bayes_pipeline.joblib
Da luu mo hinh Logistic Regression: C:\ML\BTL\sms-spam-detection-converted\models\logistic_regression_pipeline.joblib
naive_bayes: spam | Ham=0.0851 | Spam=0.9149
logistic_regression: spam | Ham=0.1229 | Spam=0.8771


## 7. Chạy ứng dụng

Sau khi Notebook chạy thành công, mở Terminal tại thư mục gốc và chạy `streamlit run app.py`. Tab **Detection** tải các mô hình trong `models/`; tab **Performance** sử dụng `outputs/metrics.csv` và ba biểu đồ PNG.